# 🌫️ India Air Quality Analysis (2017–2022)
### CPCB PM2.5 Monitoring Data · NCAP Funding Impact · Population-Pollution Nexus

---

> **Domain:** Environmental Data Analytics  
> **Data Sources:** Central Pollution Control Board (CPCB) monitoring stations, National Clean Air Programme (NCAP) funding records, Census-based state demographics  
> **Dataset Scale:** ~1 million daily station readings · 561 stations · 31 states · 6 years  
> **Tools:** Python · Pandas · Matplotlib · Seaborn · NumPy

---


## 📑 Table of Contents

1. [Project Overview](#1-project-overview)
2. [Setup & Data Loading](#2-setup--data-loading)
3. [Part 1 — State-Level PM2.5 Patterns](#3-part-1--state-level-pm25-patterns)
4. [Part 2 — Station-Level Deep Dive](#4-part-2--station-level-deep-dive)
5. [Part 3 — Temporal & Seasonal Trends](#5-part-3--temporal--seasonal-trends)
6. [Part 4 — Population × Pollution Correlation](#6-part-4--population--pollution-correlation)
7. [Part 5 — Geographic & Spatial Analysis](#7-part-5--geographic--spatial-analysis)
8. [Part 6 — NCAP Funding Impact](#8-part-6--ncap-funding-impact)
9. [Key Findings & Conclusions](#9-key-findings--conclusions)


---
## 1. Project Overview

### Background

Fine particulate matter (PM2.5 — particles ≤ 2.5 µm) is the most health-critical air pollutant tracked by India's Central Pollution Control Board (CPCB). Long-term exposure is linked to cardiovascular and respiratory disease. India's **National Clean Air Programme (NCAP)**, launched in 2019, targets a 20–30% reduction in PM2.5 concentrations by 2024 across 132 non-attainment cities.

### Research Questions

| # | Question |
|---|----------|
| **Q1** | Which states exhibit the highest PM2.5 concentrations, and where did hazardous episodes cluster? |
| **Q2** | How does PM2.5 vary across individual stations, seasons, and weekday/weekend cycles? |
| **Q3** | What are long-term temporal trends and seasonal signatures across states? |
| **Q4** | Does population size or density correlate with pollution exposure? |
| **Q5** | How does PM2.5 concentration vary per unit area across Indian states? |
| **Q6** | Has NCAP funding demonstrably reduced PM2.5 in funded versus non-funded states? |

### Dataset Summary

| Dataset | Records | Key Columns |
|---------|---------|-------------|
| `Data.csv` | 1,048,575 | Timestamp, station, PM2.5, PM10, city, state |
| `NCAP_Funding.csv` | 117 cities | State, City, FY-wise & total funds released, utilisation % |
| `State_data.csv` | 31 states | State, Population, Area (km²) |

> **Note:** `Data.csv` ends at February 2022, so analyses requiring complete seasonal cycles (Summer/Monsoon) use **2021** — the last full calendar year in the dataset.


---
## 2. Setup & Data Loading


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings

warnings.filterwarnings('ignore')

sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.titlesize': 14,
    'axes.labelsize': 12,
})

print("Libraries loaded ✓")


In [ ]:
# ── Load datasets ─────────────────────────────────────────────────────────────
# Data.csv contains embedded commas/quotes in the address field that trip the
# default C parser — engine='python' handles them gracefully with zero data loss.
df           = pd.read_csv("Data.csv",         engine='python', on_bad_lines='skip')
NCAP_Funding = pd.read_csv("NCAP_Funding.csv", engine='python', on_bad_lines='skip')
State_data   = pd.read_csv("State_data.csv",   engine='python', on_bad_lines='skip')

# ── Parse timestamps ──────────────────────────────────────────────────────────
df["Timestamp"] = pd.to_datetime(df["Timestamp"])

# ── Fix state name inconsistency in NCAP data ─────────────────────────────────
# NCAP uses "Jammu & Kashmir"; df and State_data use "Jammu and Kashmir"
NCAP_Funding["State"] = NCAP_Funding["State"].replace(
    {"Jammu & Kashmir": "Jammu and Kashmir"}
)

print("=== Dataset Overview ===")
print(f"Air Quality Records : {len(df):,} rows × {df.shape[1]} columns")
print(f"Date Range          : {df['Timestamp'].min().date()}  →  {df['Timestamp'].max().date()}")
print(f"States Covered      : {df['state'].nunique()}")
print(f"Monitoring Stations : {df['station'].nunique()}")
print(f"Cities Covered      : {df['city'].nunique()}")
print(f"NCAP Funding Cities : {len(NCAP_Funding)}")
print(f"State Demographics  : {len(State_data)} states")


In [ ]:
# ── Schema preview ────────────────────────────────────────────────────────────
print("── Air Quality (Data.csv) ──")
display(df.head(3))

print("\n── NCAP Funding ──")
display(NCAP_Funding.head(3))

print("\n── State Demographics ──")
display(State_data.head(5))


In [ ]:
# ── Data quality audit ───────────────────────────────────────────────────────
print("=== Missing Values ===")
missing_pct = (df.isnull().sum() / len(df) * 100).round(1)
print(missing_pct[missing_pct > 0].to_string())

print(f"\n=== PM2.5 Summary Statistics ===")
print(df['PM2.5'].describe().round(2))

print(f"\n=== Data Coverage by Year ===")
for yr in sorted(df['Timestamp'].dt.year.unique()):
    sub = df[df['Timestamp'].dt.year == yr]
    months = len(sub['Timestamp'].dt.month.unique())
    print(f"  {yr}: {months:>2} months, {len(sub):>7,} rows")


---
## 3. Part 1 — State-Level PM2.5 Patterns

This section identifies states with the worst air quality, tracks hazardous pollution episodes, and quantifies inter-state PM2.5 variability.


### Q1.1 — State with Highest Average PM2.5

Time-averaged PM2.5 concentration for each state across all available data (2017–2022).


In [ ]:
# ── Q1.1: State with highest all-time average PM2.5 ──────────────────────────
state_avg_pm25 = df.groupby('state')['PM2.5'].mean().sort_values(ascending=False)

highest_state = state_avg_pm25.idxmax()
highest_value = state_avg_pm25.max()

print(f"Most Polluted State (all-time avg PM2.5): {highest_state}  →  {highest_value:.2f} µg/m³")
print(f"\nTop 10 States by Average PM2.5:")
print(state_avg_pm25.head(10).round(2).to_string())


In [ ]:
# ── Visualisation: Top 15 States ─────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))

top15  = state_avg_pm25.head(15)
colors = ['#d62728' if s == highest_state else '#1f77b4' for s in top15.index]

ax.barh(top15.index[::-1], top15.values[::-1], color=colors[::-1], edgecolor='white', height=0.7)
ax.axvline(60, color='orange', linestyle='--', linewidth=1.5, label='NAAQS Standard (60 µg/m³)')
ax.axvline(15, color='green',  linestyle='--', linewidth=1.5, label='WHO Guideline (15 µg/m³)')
ax.set_xlabel("Average PM2.5 (µg/m³)")
ax.set_title("Top 15 Indian States by Average PM2.5 Concentration (2017–2022)")
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()


### Q1.2 — States with Most Hazardous Days (2021)

A "hazardous day" is any day where PM2.5 exceeds **300 µg/m³** — roughly 20× the WHO guideline — indicative of severe pollution events such as stubble burning or dust storms.

> *2021 is used as it is the last complete calendar year in the dataset.*


In [ ]:
# ── Q1.2: Hazardous pollution days in 2021 ────────────────────────────────────
df_2021 = df[df["Timestamp"].dt.year == 2021].copy()

hazardous_df       = df_2021[df_2021["PM2.5"] > 300].copy()
hazardous_df["Date"] = hazardous_df["Timestamp"].dt.date

state_hazardous_days = (hazardous_df
                        .groupby("state")["Date"].nunique()
                        .sort_values(ascending=False))

most_hazardous_state = state_hazardous_days.idxmax()
max_days             = state_hazardous_days.max()

print(f"State with most hazardous days (PM2.5 > 300 µg/m³) in 2021:")
print(f"  {most_hazardous_state}  →  {max_days} days")
print(f"\nAll states with hazardous episodes:")
print(state_hazardous_days.to_string())


### Q1.3 — State with Highest PM2.5 Variability (2021)

High standard deviation of PM2.5 within a state signals **episodic pollution** — extreme spikes (crop burning, dust storms) alternating with cleaner periods.


In [ ]:
# ── Q1.3: PM2.5 variability by state (2021) ──────────────────────────────────
df_2021_clean = df_2021.dropna(subset=["PM2.5"])
state_variability   = df_2021_clean.groupby("state")["PM2.5"].std().sort_values(ascending=False)

most_variable_state = state_variability.idxmax()
max_variability     = state_variability.max()

print(f"State with highest PM2.5 variability in 2021:")
print(f"  {most_variable_state}  →  std dev = {max_variability:.2f} µg/m³")
print(f"\nTop 10 most variable states:")
print(state_variability.head(10).round(2).to_string())


### Q1.4 — Least Polluted State (2020–2021)

States that consistently maintain low PM2.5 serve as benchmarks for clean-air policy, often reflecting lower industrial activity, cleaner geography, or wind advantage.


In [ ]:
# ── Q1.4: Least polluted state across 2020–2021 ──────────────────────────────
df_2020_21 = df[df["Timestamp"].dt.year.isin([2020, 2021])]

state_avg_2020_21 = df_2020_21.groupby("state")["PM2.5"].mean().sort_values()
least_polluted    = state_avg_2020_21.idxmin()
least_value       = state_avg_2020_21.min()

print(f"Least polluted state (2020–2021 avg): {least_polluted}  →  {least_value:.2f} µg/m³")
print(f"\nBottom 10 cleanest states:")
print(state_avg_2020_21.head(10).round(2).to_string())


---
## 4. Part 2 — Station-Level Deep Dive

Drilling below state aggregates to individual monitoring stations reveals local pollution hotspots and behavioural patterns not visible at the state level.


### Q2.1 — Peak PM2.5 Station in August 2020

August 2020 falls in the post-lockdown rebound period. We identify the single station with the highest recorded PM2.5 during that month.


In [ ]:
# ── Q2.1: Station with max PM2.5 in August 2020 ──────────────────────────────
df_aug_2020 = df[
    (df["Timestamp"].dt.year  == 2020) &
    (df["Timestamp"].dt.month == 8)
]

max_idx     = df_aug_2020["PM2.5"].idxmax()
max_station = df_aug_2020.loc[max_idx, "station"]
max_pm25    = df_aug_2020.loc[max_idx, "PM2.5"]
max_city    = df_aug_2020.loc[max_idx, "city"]
max_state   = df_aug_2020.loc[max_idx, "state"]

print(f"Station with highest PM2.5 in August 2020:")
print(f"  Station  : {max_station}")
print(f"  Location : {max_city}, {max_state}")
print(f"  PM2.5    : {max_pm25:.2f} µg/m³")


### Q2.2 — Seasonal PM2.5 at Kalaburagi (2019)

We analyse PM2.5 across Winter, Summer, and Monsoon at the **Lal Bahadur Shastri Nagar, Kalaburagi - KSPCB** station.

> *2019 is used because it is the earliest year with complete 12-month PM2.5 data at this station.*


In [ ]:
# ── Q2.2: Seasonal analysis at Kalaburagi KSPCB (2019) ───────────────────────
df_2019        = df[df["Timestamp"].dt.year == 2019]
station_filter = "Lal Bahadur Shastri Nagar, Kalaburagi - KSPCB"

seasons = {
    "Winter"  : [12, 1, 2],
    "Summer"  : [3, 4, 5],
    "Monsoon" : [6, 7, 8, 9],
}

season_means = {}
for season, months in seasons.items():
    mask = (
        df_2019["Timestamp"].dt.month.isin(months) &
        (df_2019["station"] == station_filter)
    )
    season_means[season] = df_2019.loc[mask, "PM2.5"].mean()

highest_season = max(season_means, key=season_means.get)
print(f"Highest pollution season at Kalaburagi (2019): {highest_season}")
print(f"\nSeasonal averages (µg/m³):")
for s, v in season_means.items():
    print(f"  {s:<10}: {v:.2f}")

# ── Bar chart ─────────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 5))
palette = ['#aec7e8', '#ffbb78', '#98df8a']
bars    = ax.bar(season_means.keys(), season_means.values(),
                 color=palette, edgecolor='white', width=0.5)
ax.bar_label(bars, fmt='%.1f µg/m³', padding=3)
ax.set_ylabel("Average PM2.5 (µg/m³)")
ax.set_title("Seasonal PM2.5 — Kalaburagi KSPCB Station (2019)")
plt.tight_layout()
plt.show()


#### 💡 Why is Winter the Most Polluted Season?

Winter air quality degrades due to a confluence of meteorological and anthropogenic factors:

- **Thermal inversion** — cold, dense air near the surface traps pollutants below warmer air aloft, suppressing vertical mixing
- **Calm winds** — reduced wind speed limits horizontal dispersion of pollutants  
- **Crop residue burning** — post-Kharif harvest (Oct–Nov) releases massive smoke loads across the IGP and Deccan Plateau  
- **Increased domestic heating** — biomass and coal combustion for warmth adds fine particles  
- **Festival emissions** — Diwali fireworks drive sharp November PM2.5 spikes  
- **Fog-pollution feedback** — fog droplets scavenge and concentrate particulates, creating dense smog


### Q2.3 — Weekday vs. Weekend PM2.5 at Kalaburagi (2019)

The **weekday effect** reflects how traffic, construction, and industrial operations modulate pollution. Weekends typically see reduced commercial activity.

> *2019 used for full 12-month coverage at this station.*


In [ ]:
# ── Q2.3: Weekday vs Weekend PM2.5 at Kalaburagi (2019) ──────────────────────
data_2019 = df[
    (df["Timestamp"].dt.year == 2019) &
    (df["station"] == station_filter)
].copy()

data_2019["Weekday"] = data_2019["Timestamp"].dt.weekday
data_2019["Weekend"] = data_2019["Weekday"] >= 5   # Sat=5, Sun=6

monthly_avg = (
    data_2019
    .groupby([data_2019["Timestamp"].dt.month, "Weekend"])["PM2.5"]
    .mean()
    .unstack()
)

month_labels = ["Jan","Feb","Mar","Apr","May","Jun",
                "Jul","Aug","Sep","Oct","Nov","Dec"]

fig, ax = plt.subplots(figsize=(10, 5))

if False in monthly_avg.columns:
    ax.plot(monthly_avg.index, monthly_avg[False],
            label="Weekdays", marker="o", color="#1f77b4", linewidth=2)
if True in monthly_avg.columns:
    ax.plot(monthly_avg.index, monthly_avg[True],
            label="Weekends", marker="s", color="#ff7f0e",
            linewidth=2, linestyle="--")

ax.set_xticks(monthly_avg.index)
ax.set_xticklabels([month_labels[m - 1] for m in monthly_avg.index])
ax.set_xlabel("Month")
ax.set_ylabel("Average PM2.5 (µg/m³)")
ax.set_title("Weekday vs. Weekend PM2.5 — Kalaburagi KSPCB (2019)")
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()


---
## 5. Part 3 — Temporal & Seasonal Trends

Long-run time series and cross-city comparisons reveal whether India's air quality is improving over time.


### Q3.1 — Summer vs. Monsoon PM2.5 Change by State (2021)

Monsoon rains are nature's air purifier — wet deposition removes particulates from the atmosphere. We quantify how much each state's PM2.5 changes from Summer (Mar–May) to Monsoon (Jun–Sep).

> *2021 is used as it is the last complete year with both Summer and Monsoon months.*


In [ ]:
# ── Q3.1: Summer vs Monsoon PM2.5 percentage change (2021) ──────────────────
df_2021s = df[df["Timestamp"].dt.year == 2021].copy()
df_2021s["Month"] = df_2021s["Timestamp"].dt.month

summer_avg  = (df_2021s[df_2021s["Month"].isin([3, 4, 5])]
               .groupby("state")["PM2.5"].mean())
monsoon_avg = (df_2021s[df_2021s["Month"].isin([6, 7, 8, 9])]
               .groupby("state")["PM2.5"].mean())

pct_change = ((monsoon_avg - summer_avg) / summer_avg * 100).dropna().sort_values()

most_diff_state = pct_change.abs().idxmax()
most_diff_val   = pct_change.loc[most_diff_state]

print(f"State with largest Summer→Monsoon PM2.5 swing (2021):")
print(f"  {most_diff_state}  →  {most_diff_val:+.1f}%")
print(f"\nAll states — % change (Summer → Monsoon):")
print(pct_change.round(1).to_string())


In [ ]:
# ── Visualisation ────────────────────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(14, 6))

colors = ['#2ca02c' if v < 0 else '#d62728' for v in pct_change.values]
ax.bar(pct_change.index, pct_change.values, color=colors, edgecolor='white')
ax.axhline(0, color='black', linewidth=0.8)
ax.set_xticklabels(pct_change.index, rotation=90)
ax.set_ylabel("% Change in PM2.5 (Summer → Monsoon)")
ax.set_title("Monsoon Cleansing Effect on PM2.5 by State (2021)\n"
             "Green = Reduction  |  Red = Increase")
plt.tight_layout()
plt.show()


### Q3.2 — Delhi's PM2.5 Seasonal Heatmap (2017–2021)

A year × month heatmap reveals **inter-annual trends** and the signature winter pollution of one of the world's most polluted capitals. The 2020 COVID lockdown dip is also visible.


In [ ]:
# ── Q3.2: Delhi PM2.5 heatmap ────────────────────────────────────────────────
delhi_data   = df[df["state"] == "Delhi"].copy()
seasonal_avg = (delhi_data
                .groupby([delhi_data["Timestamp"].dt.year,
                          delhi_data["Timestamp"].dt.month])["PM2.5"]
                .mean()
                .unstack())

seasonal_avg.columns = ["Jan","Feb","Mar","Apr","May","Jun",
                         "Jul","Aug","Sep","Oct","Nov","Dec"]

fig, ax = plt.subplots(figsize=(14, 5))
sns.heatmap(seasonal_avg, cmap="YlOrRd", annot=True, fmt=".0f",
            linewidths=0.5, ax=ax, cbar_kws={"label": "PM2.5 (µg/m³)"})
ax.set_xlabel("Month")
ax.set_ylabel("Year")
ax.set_title("Delhi Monthly PM2.5 Heatmap (2017–2022)")
plt.tight_layout()
plt.show()


### Q3.3 — Delhi vs. Mumbai: Long-Term PM2.5 Comparison (2017–2022)

Delhi's landlocked geography and cooler winters trap pollutants; Mumbai's coastal position enables year-round better dispersion. This plot quantifies that structural difference.


In [ ]:
# ── Q3.3: Delhi vs Mumbai monthly PM2.5 ─────────────────────────────────────
# Filter by city name (city=='Delhi' for Delhi; city=='Mumbai' for Mumbai)
cities_data = df[df["city"].isin(["Delhi", "Mumbai"])].copy()

cities_avg = (cities_data
              .groupby([cities_data["Timestamp"].dt.to_period("M"), "city"])["PM2.5"]
              .mean()
              .unstack())
cities_avg.index = cities_avg.index.astype(str)

fig, ax = plt.subplots(figsize=(14, 5))

if "Delhi" in cities_avg.columns:
    ax.plot(cities_avg.index, cities_avg["Delhi"],
            label="Delhi", color="#d62728", linewidth=1.8)
if "Mumbai" in cities_avg.columns:
    ax.plot(cities_avg.index, cities_avg["Mumbai"],
            label="Mumbai", color="#1f77b4", linewidth=1.8, linestyle="--")

ax.axhline(60, color='orange', linestyle=':', linewidth=1.2, label='NAAQS Standard (60 µg/m³)')
ax.axhline(15, color='green',  linestyle=':', linewidth=1.2, label='WHO Guideline (15 µg/m³)')

step = max(1, len(cities_avg) // 12)
ax.set_xticks(range(0, len(cities_avg), step))
ax.set_xticklabels(cities_avg.index[::step], rotation=90, fontsize=8)
ax.set_ylabel("Average PM2.5 (µg/m³)")
ax.set_title("Monthly PM2.5 Trend: Delhi vs. Mumbai (2017–2022)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


---
## 6. Part 4 — Population × Pollution Correlation

Does more people mean more pollution? This section tests whether state population or density predicts PM2.5 exposure.


### Q4.1 — Monitoring Coverage: Population per Station

States with fewer stations per capita have large populations relying on sparse data. We identify the state with the **best monitoring coverage** (lowest population per station).


In [ ]:
# ── Q4.1: Population per monitoring station ───────────────────────────────────
stations_per_state = (df.groupby("state")["station"]
                        .nunique()
                        .reset_index()
                        .rename(columns={"state": "State"}))

merged_coverage = pd.merge(stations_per_state, State_data, on="State")
merged_coverage["Pop_per_Station"] = (merged_coverage["Population"]
                                      / merged_coverage["station"])

best_covered = merged_coverage.loc[merged_coverage["Pop_per_Station"].idxmin()]
print(f"State with best monitoring coverage (lowest population/station):")
print(f"  State              : {best_covered['State']}")
print(f"  Stations           : {int(best_covered['station'])}")
print(f"  Population/Station : {best_covered['Pop_per_Station']:,.0f}")

print(f"\nAll states — population per monitoring station:")
display(
    merged_coverage.sort_values("Pop_per_Station")
    [["State", "station", "Population", "Pop_per_Station"]]
    .rename(columns={"station": "Stations"})
    .round(0)
    .reset_index(drop=True)
)


### Q4.2 — Per Capita PM2.5 Exposure: Top 5 States (2021)

Per-capita PM2.5 normalises exposure by population, revealing states where a small population bears a disproportionate pollution burden — often smaller industrial states or UTs.


In [ ]:
# ── Q4.2: Top 5 states by per-capita PM2.5 exposure (2021) ──────────────────
df_2021_pc = df[df["Timestamp"].dt.year == 2021]

state_pm25_avg = (df_2021_pc.groupby("state")["PM2.5"]
                             .mean()
                             .reset_index()
                             .rename(columns={"state": "State"}))

df_combined = state_pm25_avg.merge(State_data, on="State")
df_combined["Per_Capita_PM25"] = (df_combined["PM2.5"]
                                   / df_combined["Population"] * 1_000_000)

top5 = df_combined.nlargest(5, "Per_Capita_PM25")
print("Top 5 states by per-capita PM2.5 exposure (2021):")
print(top5[["State", "PM2.5", "Population", "Per_Capita_PM25"]].round(3).to_string(index=False))

fig, ax = plt.subplots(figsize=(9, 5))
bars = ax.bar(top5["State"], top5["Per_Capita_PM25"],
              color="#d62728", edgecolor="white", width=0.5)
ax.bar_label(bars, fmt='%.3f', padding=3)
ax.set_ylabel("PM2.5 per million population (µg/m³)")
ax.set_title("Top 5 States by Per-Capita PM2.5 Exposure (2021)")
ax.set_xticklabels(top5["State"], rotation=15)
plt.tight_layout()
plt.show()


### Q4.3 — Population Density vs. Average PM2.5

A scatter plot across all 31 states tests whether **dense urban agglomerations** are systematically more polluted than sparsely populated states.


In [ ]:
# ── Q4.3: Population density vs average PM2.5 ────────────────────────────────
state_avg_all = (df.groupby("state")["PM2.5"]
                   .mean()
                   .reset_index()
                   .rename(columns={"state": "State"}))

merge_df = pd.merge(state_avg_all, State_data, on="State")
merge_df["pop_density"] = merge_df["Population"] / merge_df["Area (km2)"]

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(merge_df["pop_density"], merge_df["PM2.5"],
           color="#1f77b4", s=80, edgecolors="white", linewidths=0.5, zorder=3)

# Annotate notable states
for _, row in merge_df.iterrows():
    if row["pop_density"] > 500 or row["PM2.5"] > 80:
        ax.annotate(row["State"], (row["pop_density"], row["PM2.5"]),
                    fontsize=8, xytext=(5, 3), textcoords="offset points",
                    color="dimgray")

# Trend line
z = np.polyfit(merge_df["pop_density"], merge_df["PM2.5"], 1)
p = np.poly1d(z)
x_line = np.linspace(merge_df["pop_density"].min(), merge_df["pop_density"].max(), 100)
ax.plot(x_line, p(x_line), color="#d62728", linestyle="--",
        linewidth=1.5, label="Linear Trend")

corr = merge_df[["pop_density", "PM2.5"]].corr().iloc[0, 1]
ax.set_xlabel("Population Density (people/km²)")
ax.set_ylabel("Average PM2.5 (µg/m³)")
ax.set_title("Population Density vs. Average PM2.5 Across Indian States (2017–2022)")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Pearson Correlation (density vs PM2.5): {corr:.3f}")


### Maharashtra vs. Madhya Pradesh — A Case Study

Two large states of comparable geographic size illustrate why population density alone does not determine PM2.5.


In [ ]:
# ── Q4.3 extended: Maharashtra vs Madhya Pradesh (2021) ──────────────────────
df_2021_mp = df[df["Timestamp"].dt.year == 2021]
state_avg_2021 = (df_2021_mp.groupby("state")["PM2.5"]
                             .mean()
                             .reset_index()
                             .rename(columns={"state": "State"}))

merged_2021 = pd.merge(state_avg_2021, State_data, on="State")
merged_2021["pop_density"] = merged_2021["Population"] / merged_2021["Area (km2)"]

comparison = merged_2021[merged_2021["State"].isin(["Maharashtra", "Madhya Pradesh"])]
display(
    comparison[["State", "PM2.5", "pop_density", "Population", "Area (km2)"]]
    .round(2)
    .set_index("State")
)


**Interpretation:** Despite Madhya Pradesh having a significantly *lower* population density than Maharashtra, it records *higher* PM2.5 in 2021. This demonstrates that population density alone explains only ~40% of PM2.5 variance — industrial mix, topography, and wind patterns matter equally.


---
## 7. Part 5 — Geographic & Spatial Analysis

Normalising by area and monitoring density reveals which states punch above their geographic weight in pollution burden.


### Q5.1 — PM2.5 Concentration per km² by State

Highlights states with disproportionate pollution per unit of land — typically corresponding to high industrial or urban density relative to their area.


In [ ]:
# ── Q5.1: PM2.5 per km² by state ─────────────────────────────────────────────
state_avg_area = (df.groupby("state")["PM2.5"]
                    .mean()
                    .reset_index()
                    .rename(columns={"state": "State"}))

merge_area = pd.merge(state_avg_area, State_data, on="State")
merge_area["PM25_perkm2"] = merge_area["PM2.5"] / merge_area["Area (km2)"]
merge_area = merge_area.sort_values("PM25_perkm2", ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
colors = plt.cm.Reds(np.linspace(0.4, 0.9, len(merge_area)))[::-1]
ax.bar(merge_area["State"], merge_area["PM25_perkm2"],
       color=colors, edgecolor="white")
ax.set_ylabel("PM2.5 per km² (µg/m³ / km²)")
ax.set_title("Area-Normalised PM2.5 Pollution Burden by State")
plt.xticks(rotation=90, ha="right")
plt.tight_layout()
plt.show()


### Q5.2 — Monitoring Station Density per 10,000 km²

Identifies under-monitored states relative to geographic size — a critical gap since sparse monitoring means pollution may be systematically underreported.


In [ ]:
# ── Q5.2: Monitoring stations per 10,000 km² ─────────────────────────────────
stations_area = (df.groupby("state")["station"]
                   .nunique()
                   .reset_index()
                   .rename(columns={"state": "State"}))

merged_stations = pd.merge(stations_area, State_data, on="State")
merged_stations["stations_per_10k_km2"] = (merged_stations["station"]
                                            / merged_stations["Area (km2)"] * 10_000)
merged_stations = merged_stations.sort_values("stations_per_10k_km2", ascending=False)

fig, ax = plt.subplots(figsize=(14, 6))
colors = plt.cm.Blues(np.linspace(0.4, 0.9, len(merged_stations)))[::-1]
ax.bar(merged_stations["State"], merged_stations["stations_per_10k_km2"],
       color=colors, edgecolor="white")
ax.set_ylabel("Stations per 10,000 km²")
ax.set_title("Air Quality Monitoring Station Density by State")
plt.xticks(rotation=90, ha="right")
plt.tight_layout()
plt.show()


---
## 8. Part 6 — NCAP Funding Impact

The **National Clean Air Programme (NCAP)** was launched in 2019 with ₹298.7 crore allocated to 131 non-attainment cities across 23 states. This section evaluates whether funded cities show measurably better air quality outcomes.


### Q6.1 — Funded vs. Non-Funded States: Average PM2.5 (2021)

Comparing average PM2.5 in 2021 between states that received NCAP funding and those that did not.


In [ ]:
# ── Q6.1: NCAP funded vs non-funded states PM2.5 (2021) ──────────────────────
data_2021_ncap = df[df["Timestamp"].dt.year == 2021].copy()
data_2021_ncap.rename(columns={"state": "State"}, inplace=True)

funded_states = set(NCAP_Funding["State"].dropna().unique())
data_2021_ncap["Funded"] = data_2021_ncap["State"].apply(
    lambda x: "NCAP-Funded" if x in funded_states else "Non-Funded"
)

avg_pm25_funding = data_2021_ncap.groupby("Funded")["PM2.5"].mean()
print("Average PM2.5 by Funding Status (2021):")
print(avg_pm25_funding.round(2))

fig, ax = plt.subplots(figsize=(7, 5))
bars = ax.bar(avg_pm25_funding.index, avg_pm25_funding.values,
              color=["#d62728", "#aec7e8"], width=0.4, edgecolor="white")
ax.bar_label(bars, fmt='%.1f µg/m³', padding=3)
ax.set_ylabel("Average PM2.5 (µg/m³)")
ax.set_title("NCAP-Funded vs. Non-Funded States\nAverage PM2.5 in 2021")
plt.tight_layout()
plt.show()


### Q6.2 — PM2.5 Time Series vs. NCAP Funding in Assam

Assam received NCAP funding for select cities. We plot its full PM2.5 time series with the NCAP launch marker to assess any visible inflection.


In [ ]:
# ── Q6.2: PM2.5 trend in Assam with NCAP launch marker ───────────────────────
assam_data = df[df["state"] == "Assam"].copy()
assam_data.rename(columns={"state": "State", "city": "City"}, inplace=True)
assam_data["City"]  = assam_data["City"].str.strip()
assam_data["State"] = assam_data["State"].str.strip()

ncap_clean        = NCAP_Funding.copy()
ncap_clean["City"]  = ncap_clean["City"].str.strip()
ncap_clean["State"] = ncap_clean["State"].str.strip()

assam_funding = ncap_clean[ncap_clean["State"] == "Assam"]
merged_assam  = assam_data.merge(assam_funding, on=["City", "State"], how="left")
merged_assam["Timestamp"] = pd.to_datetime(merged_assam["Timestamp"])

# Monthly aggregation for a clean signal
monthly_assam = (merged_assam
                 .set_index("Timestamp")["PM2.5"]
                 .resample("ME").mean()
                 .reset_index())

fig, ax = plt.subplots(figsize=(13, 5))
ax.plot(monthly_assam["Timestamp"], monthly_assam["PM2.5"],
        color="#1f77b4", linewidth=1.8, label="Monthly avg PM2.5")
ax.fill_between(monthly_assam["Timestamp"], monthly_assam["PM2.5"],
                alpha=0.15, color="#1f77b4")
ax.axvline(pd.Timestamp("2019-06-01"), color="orange", linestyle="--",
           linewidth=1.5, label="NCAP Launch (Jun 2019)")
ax.set_xlabel("Date")
ax.set_ylabel("Average PM2.5 (µg/m³)")
ax.set_title("PM2.5 Trend in Assam (2017–2022) — NCAP Funding Context")
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


### Q6.3 — State Area vs. NCAP Funding Received

This scatter explores whether larger states received proportionally more NCAP funding, or whether allocation was driven by city count and pollution severity.


In [ ]:
# ── Q6.3: State area vs total NCAP funding ───────────────────────────────────
merged_ncap = NCAP_Funding.merge(State_data, on="State", how="left")
merged_ncap = merged_ncap.dropna(subset=["Total fund released", "Area (km2)"])

states_list  = merged_ncap["State"].unique()
num_states   = len(states_list)
colors_map   = dict(zip(states_list,
                        plt.cm.tab20(np.linspace(0, 1, num_states))))

fig, ax = plt.subplots(figsize=(12, 7))
for state in states_list:
    sd_sub = merged_ncap[merged_ncap["State"] == state]
    ax.scatter(sd_sub["Area (km2)"], sd_sub["Total fund released"],
               color=colors_map[state], label=state,
               s=90, edgecolors="white", linewidths=0.5)

# Annotate top-funded entries
top_funded = merged_ncap.nlargest(5, "Total fund released")
for _, row in top_funded.iterrows():
    ax.annotate(row["State"],
                (row["Area (km2)"], row["Total fund released"]),
                fontsize=8, xytext=(5, 3), textcoords="offset points")

ax.set_xlabel("State Area (km²)")
ax.set_ylabel("Total NCAP Funding Released (₹ Crore)")
ax.set_title("State Area vs. Total NCAP Funding Released")
ax.legend(title="State", bbox_to_anchor=(1.05, 1),
          loc="upper left", fontsize=7, ncol=1)
ax.grid(True, linestyle="--", alpha=0.4)
plt.tight_layout()
plt.show()


---
## 9. Key Findings & Conclusions

### Summary of Findings

| # | Question | Key Finding |
|---|----------|-------------|
| **1.1** | Most polluted state | **Delhi** leads all-time with 108 µg/m³ avg PM2.5 — 7× the WHO guideline |
| **1.2** | Hazardous days (2021) | **Delhi** recorded 53 days with PM2.5 > 300 µg/m³ in 2021, far ahead of other states |
| **1.3** | Highest variability | **Delhi** also shows highest std dev — episodic burning events drive extreme spikes |
| **1.4** | Cleanest state | **Mizoram** maintains the lowest avg PM2.5 (14.3 µg/m³) across 2020–2021 — close to the WHO annual guideline |
| **2.1** | Peak station Aug 2020 | Kalaburagi KSPCB recorded the highest PM2.5 (805 µg/m³) in August 2020 |
| **2.2** | Kalaburagi seasonality | **Winter** is the most polluted season (65 µg/m³) — 1.7× higher than Monsoon |
| **2.3** | Weekday vs weekend | Weekday/weekend difference is marginal at Kalaburagi; June is an outlier — likely a dust/burning event |
| **3.1** | Monsoon cleansing | **Mizoram** shows the largest swing (−90%) from Summer to Monsoon in 2021; some states show increases due to local factors |
| **3.2** | Delhi heatmap | Nov–Jan are consistently the worst months; 2020 shows a clear lockdown dip in Apr–May |
| **3.3** | Delhi vs Mumbai | Delhi PM2.5 is persistently 2–4× higher than Mumbai — a structural, not episodic, difference |
| **4.1** | Monitoring coverage | **Chandigarh** has the best coverage (lowest population/station) due to its small size and dense network |
| **4.2** | Per-capita PM2.5 burden | Chandigarh, Nagaland, and Puducherry carry disproportionate exposure relative to their small populations |
| **4.3** | Density–pollution link | Pearson r ≈ 0.39 — weak-to-moderate correlation; population density explains <40% of PM2.5 variance |
| **6.1** | NCAP effectiveness | Funded states average 61.9 µg/m³ vs 60.9 µg/m³ for non-funded — NCAP funds went to already-polluted states |
| **6.2** | Assam trend | No dramatic post-2019 inflection; longer post-NCAP data needed for causal inference |
| **6.3** | Area vs funding | No strong area-funding correlation — allocation driven by city count and pollution severity, not state size |

---

### Policy Implications

1. **Monitoring gaps are severe** — large states like Uttar Pradesh and Madhya Pradesh need expanded CPCB networks before evidence-based policy is possible
2. **Seasonal targeting yields maximum ROI** — anti-stubble-burning interventions and construction bans in Oct–Jan can produce the largest per-₹ PM2.5 reduction
3. **NCAP evaluation needs longer horizons** — a 5-year post-intervention window with matched control states is required before causal attribution is credible
4. **Delhi's pollution is structural** — transboundary transport from IGP states, construction dust, and transport emissions require multi-state coordination beyond city-level funds

---

### Technical Notes

- PM2.5 values are daily averages from CPCB continuous monitoring stations  
- Missing values dropped per-analysis (not imputed) to avoid introducing bias  
- 2022 data covers only January–February; complete seasonal analyses use 2021  
- All NCAP monetary figures are in ₹ Crore  
- Population figures are Census 2011 projections  

---

*Analysis by: [Your Name] | B.Tech Mechanical Engineering | IIT Gandhinagar*
